In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import os
import base64
from datetime import datetime

# --- FILE SETUP ---
FILE = "bill_history.csv"
if os.path.exists(FILE):
    history_df = pd.read_csv(FILE)
else:
    # Initializing with Share column to prevent NaN errors
    history_df = pd.DataFrame(columns=["Date", "Bill", "Total", "Share", "Result", "Comments"])

# --- THEME ENGINE ---
style_output = widgets.Output()

def get_style(dark=True):
    # Palette
    bg = "#1a202c" if dark else "#ffffff"
    text = "#f7fafc" if dark else "#1a202c"
    card = "#2d3748" if dark else "#f1f5f9"
    border = "#4a5568" if dark else "#cbd5e1"
    
    return HTML(f"""
    <style>
        /* Force background on every possible container layer in Binder/Voila */
        :root, body, .voila-app, .jp-Notebook, .widget-subarea, 
        #jp-main-content-panel, .output_wrapper, .v-application, .jp-Cell {{ 
            background-color: {bg} !important; 
            min-height: 100vh !important;
            color: {text} !important;
        }}

        /* Label Fix: Ensure they are visible against the forced background */
        .widget-label, label, .widget-html-content, h2, h3, .jp-RenderedHTMLCommon {{ 
            color: {text} !important; 
            min-width: 140px !important; 
            font-weight: 600 !important;
            opacity: 1 !important;
        }}

        /* History Table: Forced row visibility and alignment */
        .dataframe {{ 
            background-color: {card} !important; 
            color: {text} !important; 
            width: 100% !important; 
            border-collapse: collapse !important;
        }}
        .dataframe th {{ 
            background-color: {border} !important; 
            color: {text} !important; 
            padding: 10px !important;
            text-align: left !important;
        }}
        .dataframe td {{ 
            padding: 12px 10px !important; 
            border-bottom: 1px solid {border} !important;
            color: {text} !important; /* Forced visibility for rows */
            opacity: 1 !important;
            text-align: left !important;
        }}

        /* Summary Box */
        .summary-box {{
            background-color: {card}; 
            border-left: 5px solid #48bb78;
            padding: 15px; margin-bottom: 20px; border-radius: 8px;
            color: {text} !important;
            border: 1px solid {border};
        }}
        
        /* Input Field Visibility */
        input, textarea, .widget-text input, .widget-textarea textarea {{
            background-color: {card} !important;
            color: {text} !important;
            border: 1px solid {border} !important;
        }}
        
        .jupyter-button {{ min-width: 160px !important; height: 40px !important; }}

        /* --- NEW: MODAL OVERLAY FIX --- */
        .modal-container {{
            position: fixed;
            top: 0; left: 0;
            width: 100%; height: 100%;
            background: rgba(0,0,0,0.85); /* Darken the background */
            z-index: 10000;
            display: flex;
            justify-content: center;
            align-items: center;
        }}

        .modal-box {{
            background: white !important;
            padding: 40px;
            border-radius: 15px;
            text-align: center;
            border: 3px solid #ef4444;
            color: #1a202c !important; /* Force dark text on white box */
            z-index: 10001; 
            box-shadow: 0 10px 40px rgba(0,0,0,0.8);
        }}
    </style>
    """)

In [113]:
# 1. THEME TOGGLE
theme_toggle = widgets.ToggleButton(
    value=True, 
    description='🌙 Dark Mode', 
    button_style='info', 
    icon='moon',
    layout=widgets.Layout(width='150px')
)

# 2. INPUT WIDGETS
title_input = widgets.Text(description="Bill Name:", placeholder="e.g. Electric Bill")
comment_input = widgets.Textarea(
    description="Comments:", 
    placeholder="Optional details...", 
    layout=widgets.Layout(height='60px')
)
andrew_input = widgets.FloatText(description="Andrew Paid ($):", value=0.0)
moi_input = widgets.FloatText(description="Moi Paid ($):", value=0.0)

# 3. ACTION BUTTONS
calc_button = widgets.Button(description="Calculate & Save", button_style='success')
delete_button = widgets.Button(description="Delete Last", button_style='danger')

# 4. MODAL ELEMENTS
confirm_yes = widgets.Button(description="Yes, Delete It", button_style='danger')
confirm_no = widgets.Button(description="Cancel")

# 5. OUTPUTS
summary_output = widgets.Output()
download_output = widgets.Output()
output_area = widgets.Output()
popup_output = widgets.Output()

# 6. LAYOUT ASSEMBLY
button_row = widgets.HBox([calc_button, delete_button, download_output])

header_row = widgets.HBox([
    widgets.HTML("<h2>⚖️ Bill Balancer</h2>"), 
    theme_toggle
], layout=widgets.Layout(justify_content='space-between'))

layout = widgets.VBox([
    header_row,
    summary_output, 
    title_input, 
    comment_input, 
    andrew_input, 
    moi_input, 
    widgets.HTML("<br>"),
    button_row, 
    output_area,
    popup_output
])

In [115]:
# --- 1. CORE LOGIC FUNCTIONS ---
def display_summary():
    with summary_output:
        clear_output()
        if not history_df.empty:
            try:
                current_month = datetime.now().strftime("%Y-%m")
                temp_df = history_df.copy()
                # Clean currency strings for calculation
                temp_df['Total_Num'] = temp_df['Total'].replace(r'[\$,]', '', regex=True).astype(float)
                monthly_data = temp_df[temp_df['Date'].str.startswith(current_month)]
                monthly_total = monthly_data['Total_Num'].sum()
                
                # TOTAL ENTRIES COUNTER
                total_count = len(history_df)
                
                display(HTML(f'''
                    <div class="summary-box">
                        <strong>📊 {datetime.now().strftime("%B %Y")} Summary</strong><br>
                        Total Spent: ${monthly_total:.2f}<br>
                        <small style="opacity: 0.8;">Total Entries in History: {total_count}</small>
                    </div>
                '''))
            except:
                pass

def update_history_table():
    with output_area:
        clear_output()
        if not history_df.empty:
            # Table header color logic: Blue for dark mode, dark slate for light
            h_color = '#60a5fa' if theme_toggle.value else '#1e293b'
            display(HTML(f"<h3 style='color: {h_color}; margin-top: 20px;'>📝 Recent History</h3>"))
            display(HTML(history_df.tail(10)[::-1].to_html(index=False, classes='dataframe')))

def refresh_ui():
    display_summary()
    update_history_table()
    with download_output:
        clear_output()
        if os.path.exists(FILE):
            csv_data = history_df.to_csv(index=False)
            b64 = base64.b64encode(csv_data.encode()).decode()
            display(HTML(f'<a href="data:file/csv;base64,{b64}" download="bills.csv"><button class="download-btn" style="cursor:pointer; padding:5px 10px;">📥 Download</button></a>'))

# --- 2. ACTIONS ---
def on_click(b):
    global history_df
    total = andrew_input.value + moi_input.value
    share = total / 2
    res = f"Moi owes Andrew ${andrew_input.value-share:.2f}" if andrew_input.value > share else f"Andrew owes Moi ${moi_input.value-share:.2f}" if moi_input.value > share else "Even split!"
    
    new_row = {
        "Date": datetime.now().strftime("%Y-%m-%d"), 
        "Bill": title_input.value, 
        "Total": f"${total:.2f}", 
        "Share": f"${share:.2f}", 
        "Result": res, 
        "Comments": comment_input.value
    }
    
    history_df = pd.concat([history_df, pd.DataFrame([new_row])], ignore_index=True)
    history_df.to_csv(FILE, index=False)
    
    # Reset inputs
    title_input.value = ""; comment_input.value = ""; andrew_input.value = 0.0; moi_input.value = 0.0
    refresh_ui()

def show_delete_popup(b):
    with popup_output:
        clear_output()
        display(HTML('''
            <div class="modal-container">
                <div class="modal-box">
                    <h2 style="color: #1a202c !important; margin-top:0;">⚠️ Confirm Delete?</h2>
                    <p style="color: #4a5568 !important; font-size: 1.1em;">Remove the last entry forever?</p>
                </div>
            </div>
        '''))
        display(widgets.HBox([confirm_yes, confirm_no], 
                layout={'justify_content':'center', 'margin':'-120px 0 0 0', 'position':'relative'}))

def final_delete(b):
    global history_df
    if not history_df.empty:
        history_df = history_df.drop(history_df.index[-1])
        history_df.to_csv(FILE, index=False)
        refresh_ui()
    
    # --- SUCCESS MESSAGE LOGIC ---
    with popup_output:
        clear_output()
        display(HTML('''
            <div style="position: fixed; top: 20px; right: 20px; background: #22c55e; color: white; 
                        padding: 15px 25px; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);
                        z-index: 10002; font-weight: bold; animation: fadeout 2s forwards;">
                ✅ Entry Deleted Successfully
            </div>
            <style>
                @keyframes fadeout { 0% {opacity: 1;} 70% {opacity: 1;} 100% {opacity: 0;} }
            </style>
        '''))
    
    # Clear the success message area after 2 seconds
    import threading
    threading.Timer(2.0, lambda: popup_output.clear_output()).start()

def on_theme_change(change):
    with style_output:
        clear_output()
        display(get_style(dark=change['new']))
    theme_toggle.description = '🌙 Dark Mode' if change['new'] else '☀️ Light Mode'
    refresh_ui()

# --- 3. BINDINGS & START ---
theme_toggle.observe(on_theme_change, names='value')
calc_button.on_click(on_click)
delete_button.on_click(show_delete_popup)
confirm_yes.on_click(final_delete)
confirm_no.on_click(lambda b: popup_output.clear_output())

# Start up the app
with style_output:
    display(get_style(dark=True))
display(style_output, layout)
refresh_ui()

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.HTML object>', '…